# Baseline: Sistema de Recomendação com Similaridade de Cosseno

O objetivo deste notebook é implementar o algoritmo mais simples (baseline) de recomendação baseada em conteúdo, para que possamos comparar sua performance com os futuros modelos (KNN e SVD).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Carregando os dados e a matriz X exportados no notebook 01
df = pd.read_parquet('../../Machine Learning/data/processed/books_clean.parquet')
X = np.load('../../Machine Learning/data/processed/X_matrix.npz')['X']

train_idx = np.load('../../Machine Learning/data/processed/train_idx.npy')
test_idx = np.load('../../Machine Learning/data/processed/test_idx.npy')

print(f"Base carregada! Shape de X: {X.shape}")

## 1. Função de Recomendação
Dado o índice de um livro alvo, calculamos a distância do cosseno deste livro contra todos os outros 84.000 livros da base. Retornamos os Top N mais similares.

In [ ]:
def recomendar_similares(book_idx, X_matrix, top_n=10):
    # Extrai o vetor do livro alvo (1, 1182)
    target_vector = X_matrix[book_idx].reshape(1, -1)
    
    # Calcula a similaridade do cosseno contra toda a matriz
    # Isso retorna um array de shape (1, N)
    sim_scores = cosine_similarity(target_vector, X_matrix)[0]
    
    # Pega os índices com as maiores pontuações, ignorando o próprio livro (que terá similaridade 1.0)
    # np.argsort retorna em ordem crescente, então pegamos do fim pro começo
    top_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    top_scores = sim_scores[top_indices]
    
    return top_indices, top_scores

In [ ]:
# Testando a função de recomendação
idx_teste = test_idx[0]
livro_alvo = df.iloc[idx_teste]

print(f"=== RECOMENDAÇÕES PARA: {livro_alvo['title']} (Gêneros: {livro_alvo['genre']}) ===\n")

indices_rec, scores_rec = recomendar_similares(idx_teste, X)

for i, rec_idx in enumerate(indices_rec):
    livro_rec = df.iloc[rec_idx]
    print(f"{i+1}. {livro_rec['title']} (Sim: {scores_rec[i]:.4f}) | Gêneros: {livro_rec['genre']}")

## 2. Avaliação: Precision@10
Para calcular o Precision@10 de forma rápida sem sobrecarregar a memória, vamos avaliar uma amostra do conjunto de teste. O `Precision@10` médio responderá: "Das 10 recomendações dadas, qual a porcentagem delas que pertencia aos gêneros favoritos do usuário (ou que tinham o autor/gênero exato do livro alvo)?"

In [ ]:
def calcular_precision_at_10(test_indices, X_matrix, df, n_amostras=500):
    # Usar uma amostra para evitar estourar a memória
    np.random.seed(42)
    amostra_idx = np.random.choice(test_indices, size=min(n_amostras, len(test_indices)), replace=False)
    
    precisions = []
    
    for target_idx in amostra_idx:
        indices_rec, _ = recomendar_similares(target_idx, X_matrix, top_n=10)
        
        # Critério de acerto para Content-Based: Ter pelo menos 1 gênero principal em comum
        # ou ser do mesmo autor.
        target_genres = set(df.iloc[target_idx]['genre_list'])
        
        hits = 0
        for rec_idx in indices_rec:
            rec_genres = set(df.iloc[rec_idx]['genre_list'])
            if len(target_genres.intersection(rec_genres)) > 0:
                hits += 1
                
        # Calcula precisão (hits / 10)
        precisions.append(hits / 10.0)
        
    return np.mean(precisions)

print("Calculando Precision@10 para a baseline (isso pode levar alguns segundos)...")
precision = calcular_precision_at_10(test_idx, X, df, n_amostras=1000)
print(f"\nPrecision@10 da Baseline (Cosine Similarity): {precision:.4f}")